# Variant QC

This performs variant-level QC on the sample-filtered array + srWGS ACAF variant QC datasets.

In [ ]:
%%capture
## Python Package Import
import sys
import os 
import numpy as np
import pandas as pd
from datetime import datetime

##Ensuring dsub is up to date
!pip3 install --upgrade dsub

#Defining environment variables
# Save this Python variable as an environment variable so that its easier to use within %%bash cells.
%env JOB_ID={LINE_COUNT_JOB_ID}
## Defining necessary pathways
my_bucket = os.environ['WORKSPACE_BUCKET']
project_name='HCM_GWAS_V2'
## Setting for running dsub jobs
pd.set_option('display.max_colwidth', 0)

USER_NAME = os.getenv('OWNER_EMAIL').split('@')[0].replace('.','-')

# Save this Python variable as an environment variable so that its easier to use within %%bash cells.
%env USER_NAME={USER_NAME}
%env PROJECT_NAME=HCM_GWAS_V2

## Sample Filtering

This double checks the number of individuals to be included in the sample-level filtering prior within the variant QC steps.

In [ ]:
sample_qc_filtered_ids = pd.read_csv('../3_output/1_SAMPLE_QC/overall_filtered_ids/hcm_gwas_sample_qc_pass_ids.tsv', sep='\t')
sample_qc_filtered_ids.shape

In [ ]:
ancestries = ['afr', 'eur', 'eas', 'amr', 'mid', 'sas']
ancestry_ids = [pd.read_csv(f'../3_output/1_SAMPLE_QC/overall_filtered_ids/hcm_gwas_sample_qc_pass_{x}_ids.tsv', sep='\t', header=0) for x in ancestries]
[x.shape for x in ancestry_ids]

## Array Variant QC

This has 2 steps:
1) Identify variants which pass --maf 1% and --geno 0.01 (<1% missingness) filters
2) Identify variants which deviate from HWE on ancestry-specific basis

In [ ]:
%%capture
JOB_NAME='VARIANT_QC'

# Save this Python variable as an environment variable so that its easier to use within %%bash cells.
%env JOB_NAME={JOB_NAME}
%env PHENO_FILEPATH={PHENO_FILEPATH}
%env COV_FILEPATH={COV_FILEPATH}

## Analysis Results Folder 
line_count_results_folder = os.path.join(
    os.getenv('WORKSPACE_BUCKET'),
    'dsub',
    'results',
    os.getenv('PROJECT_NAME'),
    JOB_NAME,
    datetime.now().strftime('%Y%m%d'))

## Where the output files will go
output_files = os.path.join(line_count_results_folder)

OUTPUT_FILES = output_files

# Save this Python variable as an environment variable so that its easier to use within %%bash cells.
%env OUTPUT_FILES={OUTPUT_FILES}

### Variant QC 1

In [ ]:
filename='aux_scripts/2_VARIANT_QC_array1.sh'

script = '''

set -o pipefail 
set -o errexit

plink \
    --bed "${bedfile}" \
    --bim "${bimfile}" \
    --fam "${famfile}" \
    --keep "${sample_qc_pass_ids}" \
    --maf 0.01 \
    --geno 0.01 \
    --make-just-bim \
    --memory 30000 \
    --out array_variantqc_1
    
export output_bim="array_variantqc_1.bim"

mv ${output_bim} -t ${OUTPUT_PATH}
'''

with open(filename,'w') as fp:
    fp.write(script)
    
!gsutil cp aux_scripts/2_VARIANT_QC_array1.sh {my_bucket}/dsub/scripts/{project_name}/

In [ ]:
%%bash --out LINE_COUNT_JOB_ID
#This submits the job btw for each chromosome!

# Get a shorter username to leave more characters for the job name.
DSUB_USER_NAME="$(echo "${OWNER_EMAIL}" | cut -d@ -f1)"

AOU_NETWORK="global/networks/network"
AOU_SUBNETWORK="regions/us-central1/subnetworks/subnetwork"

MACHINE_TYPE="n2-standard-8" 
SCRIPT_NAME="2_VARIANT_QC_array1.sh"
BASH_SCRIPT="${WORKSPACE_BUCKET}/dsub/scripts/HCM_GWAS_V2/${SCRIPT_NAME}"


dsub \
    --provider google-batch \
    --user-project "${GOOGLE_PROJECT}" \
    --project "${GOOGLE_PROJECT}" \
    --image "us.gcr.io/broad-dsp-gcr-public/terra-jupyter-aou:2.2.14" \
    --network "${AOU_NETWORK}" \
    --subnetwork "${AOU_SUBNETWORK}" \
    --service-account "$(gcloud config get-value account)" \
    --user "${DSUB_USER_NAME}" \
    --regions us-central1 \
    --logging "${WORKSPACE_BUCKET}/dsub/logs/{job-name}/{user-id}/$(date +'%Y%m%d/%H%M%S')/{job-id}-{task-id}-{task-attempt}.log" \
    "$@" \
    --disk-size 1000 \
    --boot-disk-size 100 \
    --machine-type ${MACHINE_TYPE} \
    --name "VAR_QC_ARRAY_1" \
    --script "${BASH_SCRIPT}" \
    --env GOOGLE_PROJECT=${GOOGLE_PROJECT} \
    --input bedfile="gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.bed" \
    --input bimfile="gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.bim" \
    --input famfile="gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.fam" \
    --input sample_qc_pass_ids="${WORKSPACE_BUCKET}/HCM_GWAS_V2/3_output/1_SAMPLE_QC/overall_filtered_ids/hcm_gwas_sample_qc_pass_ids.tsv" \
    --use-private-address \
    --output-recursive OUTPUT_PATH="${OUTPUT_FILES}"

#N.B Make sure there are no spaces after the \ otherwise the dsub script breaks

In [ ]:
%%capture
!gsutil cp {my_bucket}/dsub/results/HCM_GWAS_V2/VARIANT_QC/20251103/array_variantqc_1.bim ../3_output/2_VARIANT_QC/array/

In [ ]:
array_variantqc_1_pass = pd.read_csv('../3_output/2_VARIANT_QC/array/array_variantqc_1.bim', sep='\s+', header=None)
print(pd.read_csv('../1_input/1_array/arrays.bim', sep='\s+', header=None).shape, array_variantqc_1_pass.shape)
array_variantqc_1_pass.iloc[:,1].to_csv('../3_output/2_VARIANT_QC/array/array_variantqc_1_pass_vars.txt', index=False, sep='\t', header=None)

In [ ]:
!gsutil cp '../3_output/2_VARIANT_QC/array_variantqc_1_pass_vars.txt' {my_bucket}/HCM_GWAS_V2/3_output/2_VARIANT_QC/array/

### Variant QC 2 (HWE)

In [ ]:
filename='aux_scripts/2_VARIANT_QC_array_hwe_ancestry.sh'

script = '''

set -o pipefail 
set -o errexit

#This outputs the heterozygosity estimates for each individual
plink \
    --bed "${bedfile}" \
    --bim "${bimfile}" \
    --fam "${famfile}" \
    --keep "${sample_qc_pass_ids_ancestry}" \
    --extract "${variant_qc_1_pass_vars}" \
    --hwe 0.000001 midp \
    --make-just-bim \
    --memory 60000 \
    --out array_hwepass_"${ancestry}"
    
export output_hwe="array_hwepass_"${ancestry}".bim"

mv ${output_hwe} -t ${OUTPUT_PATH}
'''

with open(filename,'w') as fp:
    fp.write(script)
    
!gsutil cp aux_scripts/2_VARIANT_QC_array_hwe_ancestry.sh {my_bucket}/dsub/scripts/{project_name}/

In [ ]:
%%bash --out LINE_COUNT_JOB_ID
#This submits the job btw for each chromosome!

# Get a shorter username to leave more characters for the job name.
DSUB_USER_NAME="$(echo "${OWNER_EMAIL}" | cut -d@ -f1)"

AOU_NETWORK="global/networks/network"
AOU_SUBNETWORK="regions/us-central1/subnetworks/subnetwork"

SCRIPT_NAME="2_VARIANT_QC_array_hwe_ancestry.sh"
MACHINE_TYPE="n2-standard-16" 
BASH_SCRIPT="${WORKSPACE_BUCKET}/dsub/scripts/HCM_GWAS_V2/${SCRIPT_NAME}" #From above command

ancestries=('afr' 'eur' 'eas' 'amr' 'mid' 'sas')

for ANCESTRY in "${ancestries[@]}"
do
    dsub \
    --provider google-batch \
    --user-project "${GOOGLE_PROJECT}" \
    --project "${GOOGLE_PROJECT}" \
    --image "us.gcr.io/broad-dsp-gcr-public/terra-jupyter-aou:2.2.14" \
    --network "${AOU_NETWORK}" \
    --subnetwork "${AOU_SUBNETWORK}" \
    --service-account "$(gcloud config get-value account)" \
    --user "${DSUB_USER_NAME}" \
    --regions us-central1 \
    --logging "${WORKSPACE_BUCKET}/dsub/logs/{job-name}/{user-id}/$(date +'%Y%m%d/%H%M%S')/{job-id}-{task-id}-{task-attempt}.log" \
    "$@" \
    --disk-size 1000 \
    --boot-disk-size 100 \
    --machine-type ${MACHINE_TYPE} \
    --name "array_hwe_${ANCESTRY}" \
    --script "${BASH_SCRIPT}" \
    --env GOOGLE_PROJECT=${GOOGLE_PROJECT} \
    --input bedfile="gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.bed" \
    --input bimfile="gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.bim" \
    --input famfile="gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.fam" \
    --input sample_qc_pass_ids_ancestry="${WORKSPACE_BUCKET}/HCM_GWAS_V2/3_output/1_SAMPLE_QC/overall_filtered_ids/hcm_gwas_sample_qc_pass_${ANCESTRY}_ids.tsv" \
    --input variant_qc_1_pass_vars="${WORKSPACE_BUCKET}/HCM_GWAS_V2/3_output/2_VARIANT_QC/array/array_variantqc_1_pass_vars.txt" \
    --env ancestry=${ANCESTRY} \
    --use-private-address \
    --output-recursive OUTPUT_PATH="${OUTPUT_FILES}"
done

#N.B Make sure there are no spaces after the \ otherwise the dsub script breaks

In [ ]:
%%capture
!gsutil -m cp {my_bucket}/dsub/results/HCM_GWAS_V2/VARIANT_QC/20251103/array_hwepass* ../3_output/2_VARIANT_QC/array/

In [ ]:
#This outputs the variant list from the 924,561 variants after filtering out those variants which fail HWE filter in all ancestries
ancestries = ['afr', 'eur', 'eas', 'amr', 'mid', 'sas']
array_hwepass_vars = [pd.read_csv(f'../3_output/2_VARIANT_QC/array/array_hwepass_{x}.bim', sep='\s+', header=None) for x in ancestries]
array_variantqc_1_pass = pd.read_csv('../3_output/2_VARIANT_QC/array/array_variantqc_1.bim', sep='\s+', header=None)
[print(x.shape) for x in array_hwepass_vars]

In [ ]:
#This contains the array variants which fail the HWE filter on for each ancestry
array_hwefail_vars = [array_variantqc_1_pass.loc[~array_variantqc_1_pass[1].isin(x[1])] for x in array_hwepass_vars]
[print(x.shape) for x in array_hwefail_vars]

In [ ]:
#This outputs the array variants which fail HWE filter across all genetic ancestries
array_hwefail_vars_allancestries = set(array_hwefail_vars[0].iloc[:, 1]).intersection(*(set(df.iloc[:, 1]) for df in array_hwefail_vars[1:]))
len(array_hwefail_vars_allancestries)

In [ ]:
#This provides the final list of array variants which pass QC1 and per-ancestry HWE filter
array_variantqc_pass_final = array_variantqc_1_pass.loc[~array_variantqc_1_pass[1].isin(array_hwefail_vars_allancestries)]
print(array_variantqc_pass_final.shape)
array_variantqc_pass_final.iloc[:,1].to_csv('../3_output/2_VARIANT_QC/array/array_variantqc_pass_final_vars.txt', header=None, index=False, sep='\t')

In [ ]:
!gsutil -m cp -r ../3_output/2_VARIANT_QC/array/ {my_bucket}/HCM_GWAS_V2/3_output/2_VARIANT_QC/

## srWGS ACAF Variant QC

This has 2 steps:
1) Identify variants which pass --maf 0.0 (to remove monomorphic variants) and --geno 0.1 (<10% missingness) filters
2) Identify variants which deviate from HWE on ancestry-specific basis

### Variant QC 1

In [ ]:
filename='aux_scripts/2_VARIANT_QC_srWGS_ACAF_1.sh'

script = '''

set -o pipefail 
set -o errexit

plink \
    --bed "${bedfile}" \
    --bim "${bimfile}" \
    --fam "${famfile}" \
    --keep "${sample_qc_pass_ids}" \
    --maf 0.000001 \
    --geno 0.1 \
    --make-just-bim \
    --memory 30000 \
    --out srWGS_ACAF_variantqc_1_chr"${chr}"
    
export output_bim="srWGS_ACAF_variantqc_1_chr"${chr}".bim"

mv ${output_bim} -t ${OUTPUT_PATH}
'''

with open(filename,'w') as fp:
    fp.write(script)
    
!gsutil cp aux_scripts/2_VARIANT_QC_srWGS_ACAF_1.sh {my_bucket}/dsub/scripts/{project_name}/

In [ ]:
%%bash --out LINE_COUNT_JOB_ID
#This submits the job btw for each chromosome!

# Get a shorter username to leave more characters for the job name.
DSUB_USER_NAME="$(echo "${OWNER_EMAIL}" | cut -d@ -f1)"

AOU_NETWORK="global/networks/network"
AOU_SUBNETWORK="regions/us-central1/subnetworks/subnetwork"

SCRIPT_NAME="2_VARIANT_QC_srWGS_ACAF_1.sh"
MACHINE_TYPE="n2-standard-8" 
BASH_SCRIPT="${WORKSPACE_BUCKET}/dsub/scripts/HCM_GWAS_V2/${SCRIPT_NAME}"

#Iterate over chromosomes
LOWER=1
UPPER=22

for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
    dsub \
    --provider google-batch \
    --user-project "${GOOGLE_PROJECT}" \
    --project "${GOOGLE_PROJECT}" \
    --image "us.gcr.io/broad-dsp-gcr-public/terra-jupyter-aou:2.2.14" \
    --network "${AOU_NETWORK}" \
    --subnetwork "${AOU_SUBNETWORK}" \
    --service-account "$(gcloud config get-value account)" \
    --user "${DSUB_USER_NAME}" \
    --regions us-central1 \
    --logging "${WORKSPACE_BUCKET}/dsub/logs/{job-name}/{user-id}/$(date +'%Y%m%d/%H%M%S')/{job-id}-{task-id}-{task-attempt}.log" \
    "$@" \
    --disk-size 1000 \
    --boot-disk-size 100 \
    --machine-type ${MACHINE_TYPE} \
    --name "srWGS_ACAF_variantQC_1_${chromo}" \
    --script "${BASH_SCRIPT}" \
    --env GOOGLE_PROJECT=${GOOGLE_PROJECT} \
    --input bedfile="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bed" \
    --input bimfile="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bim" \
    --input famfile="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.fam" \
    --input sample_qc_pass_ids="${WORKSPACE_BUCKET}/HCM_GWAS_V2/3_output/1_SAMPLE_QC/overall_filtered_ids/hcm_gwas_sample_qc_pass_ids.tsv" \
    --env chr=${chromo} \
    --use-private-address \
    --output-recursive OUTPUT_PATH="${OUTPUT_FILES}"
done

#N.B Make sure there are no spaces after the \ otherwise the dsub script breaks

In [ ]:
!gsutil -m cp {my_bucket}/dsub/results/HCM_GWAS_V2/VARIANT_QC/20251103/srWGS_ACAF_variantqc_1* ../3_output/2_VARIANT_QC/srWGS_ACAF/

In [ ]:
#Concatenate all the variants together after passing variant QC
!awk '{print $2}' ../3_output/2_VARIANT_QC/srWGS_ACAF/*.bim > ../3_output/2_VARIANT_QC/srWGS_ACAF/srWGS_ACAF_variantqc_1_allchr.txt

In [ ]:
!wc -l ../3_output/2_VARIANT_QC/srWGS_ACAF/srWGS_ACAF_variantqc_1_allchr.txt

In [ ]:
%%capture
!gsutil cp ../3_output/2_VARIANT_QC/srWGS_ACAF/srWGS_ACAF_variantqc_1_allchr.txt {my_bucket}/HCM_GWAS_V2/3_output/2_VARIANT_QC/srWGS_ACAF/

### Variant QC 2 (HWE)

In [ ]:
filename='aux_scripts/2_VARIANT_QC_srWGS_ACAF_hwe_ancestry.sh'

script = '''

set -o pipefail 
set -o errexit

plink \
    --bed "${bedfile}" \
    --bim "${bimfile}" \
    --fam "${famfile}" \
    --keep "${sample_qc_pass_ids_ancestry}" \
    --extract "${variant_qc_1_pass_vars}" \
    --hwe 0.000001 midp \
    --make-just-bim \
    --memory 60000 \
    --out srWGS_ACAF_hwepass_"${ancestry}"_chr"${chr}"
    
export log="srWGS_ACAF_hwepass_"${ancestry}"_chr"${chr}".log"
export output_hwe="srWGS_ACAF_hwepass_"${ancestry}"_chr"${chr}".bim"

mv ${log} ${output_hwe} -t ${OUTPUT_PATH}
'''

with open(filename,'w') as fp:
    fp.write(script)
    
!gsutil cp aux_scripts/2_VARIANT_QC_srWGS_ACAF_hwe_ancestry.sh {my_bucket}/dsub/scripts/{project_name}/

In [ ]:
%%bash --out LINE_COUNT_JOB_ID
#This submits the job btw for each chromosome!

# Get a shorter username to leave more characters for the job name.
DSUB_USER_NAME="$(echo "${OWNER_EMAIL}" | cut -d@ -f1)"

AOU_NETWORK="global/networks/network"
AOU_SUBNETWORK="regions/us-central1/subnetworks/subnetwork"

SCRIPT_NAME="2_VARIANT_QC_srWGS_ACAF_hwe_ancestry.sh"
MACHINE_TYPE="n2-standard-8" 
BASH_SCRIPT="${WORKSPACE_BUCKET}/dsub/scripts/HCM_GWAS_V2/${SCRIPT_NAME}"

ancestries=('afr' 'eur' 'eas' 'amr' 'mid' 'sas')

for ANCESTRY in "${ancestries[@]}"
do 
    #Iterate over chromosomes
    LOWER=1
    UPPER=22

    for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
    do
        dsub \
        --provider google-batch \
        --user-project "${GOOGLE_PROJECT}" \
        --project "${GOOGLE_PROJECT}" \
        --image "us.gcr.io/broad-dsp-gcr-public/terra-jupyter-aou:2.2.14" \
        --network "${AOU_NETWORK}" \
        --subnetwork "${AOU_SUBNETWORK}" \
        --service-account "$(gcloud config get-value account)" \
        --user "${DSUB_USER_NAME}" \
        --regions us-central1 \
        --logging "${WORKSPACE_BUCKET}/dsub/logs/{job-name}/{user-id}/$(date +'%Y%m%d/%H%M%S')/{job-id}-{task-id}-{task-attempt}.log" \
        "$@" \
        --disk-size 1000 \
        --boot-disk-size 100 \
        --machine-type ${MACHINE_TYPE} \
        --name "srWGS_ACAF_variantQC_hwe_${ANCESTRY}_${chromo}" \
        --script "${BASH_SCRIPT}" \
        --env GOOGLE_PROJECT=${GOOGLE_PROJECT} \
        --input bedfile="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bed" \
        --input bimfile="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bim" \
        --input famfile="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.fam" \
        --input sample_qc_pass_ids_ancestry="${WORKSPACE_BUCKET}/HCM_GWAS_V2/3_output/1_SAMPLE_QC/overall_filtered_ids/hcm_gwas_sample_qc_pass_${ANCESTRY}_ids.tsv" \
        --input variant_qc_1_pass_vars="${WORKSPACE_BUCKET}/HCM_GWAS_V2/3_output/2_VARIANT_QC/srWGS_ACAF/srWGS_ACAF_variantqc_1_allchr.txt" \
        --env chr=${chromo} \
        --env ancestry=${ANCESTRY} \
        --use-private-address \
        --output-recursive OUTPUT_PATH="${OUTPUT_FILES}"
    done
done

#N.B Make sure there are no spaces after the \ otherwise the dsub script breaks

In [ ]:
%%capture
!gsutil -m cp {my_bucket}/dsub/results/HCM_GWAS_V2/VARIANT_QC/20251105/*.bim ../3_output/2_VARIANT_QC/srWGS_ACAF/hwepass/

In [ ]:
#This provides a function that runs over each of the chr.bim across all the ancestries to ID variants which fail HWE filter in all ancestries
import pandas as pd
from pathlib import Path
import os # Import os for creating dummy directories/files in example

def find_common_missing_variants(input_path, ref_filepath, file_ending, output_path):
    """
    Finds variants from a reference DataFrame that are missing from all
    .bim files in a directory that end with a specific string.
    
    Assumes the variant ID in .bim files is in the 2nd column (index 1).
    Assumes the variant ID in the ref_df is in a column named 'SNP'.
    """
    # 1. Get the set of reference variant IDs
    ref_df = pd.read_csv(ref_filepath, sep='\s+', header=None)
    ref_df=ref_df.rename(columns={1:'SNP'})
    ref_snps = set(ref_df['SNP'])
    print(f'Loaded reference i.e. variantqc_1 passed variant list for {file_ending}')
    
    # 2. Find all matching .bim files
    files = list(Path(input_path).glob(f"*{file_ending}.bim"))
#     print(files)
    
    # 3. Create a list of missing variant sets for each file
    missing_sets = [ref_snps - set(pd.read_csv(f, sep=r'\s+', header=None, usecols=[1], names=['SNP'])['SNP']) for f in files]
    
    # 4. Find the intersection of all missing sets. If no files were found, default to an empty set.
    common_missing = set.intersection(*missing_sets) if missing_sets else set()
    
    # 5. Print a warning if no files were found
    if not files:
        print(f"Warning: No files found ending in '{file_ending}.bim' in {input_path}")
        
    # 6. Define the output file path
    output_file = Path(output_path) / f"{file_ending}.txt"
    
    # 7. Ensure output directory exists
    output_file.parent.mkdir(parents=True, exist_ok=True)
    
    # 8. Write the sorted list of common missing variants to the output file
    output_file.write_text('\n'.join(sorted(list(common_missing))))
    
    # 9. Print a confirmation message
    print(f"Wrote {len(common_missing)} common missing variants to {output_file}")

bim_path = '../3_output/2_VARIANT_QC/srWGS_ACAF/hwepass/'

#Run it for each chromosome
[find_common_missing_variants(bim_path, f'../3_output/2_VARIANT_QC/srWGS_ACAF/srWGS_ACAF_variantqc_1_{suffix}.bim',suffix, bim_path) for suffix in [f"chr{i}" for i in range(1, 23)]]


In [ ]:
#Now I remove the HWE variants from each srWGS_ACAF_variantqc_1_{suffix}.bim and re_output in the qc1_hwepass_vars folder
def qc1_hwepass_vars_outputter(variantqc_1_bim_path, hwepass_txt_path, output_name,
                               output_folder='../3_output/2_VARIANT_QC/srWGS_ACAF/qc1_hwepass_vars/'):
    variantqc_1_bim_pass_vars = pd.read_csv(variantqc_1_bim_path, sep='\s+', usecols=[1], names=['SNP']) #This is the variants which pass QC 1
    hwepass_vars = pd.read_csv(hwepass_txt_path, sep='\s+', usecols=[0], names=['SNP'])
    
    out = variantqc_1_bim_pass_vars.loc[~variantqc_1_bim_pass_vars['SNP'].isin(hwepass_vars['SNP'])]
    out.to_csv(f'{output_folder}srWGS_ACAF_qc1_hwepass_vars_{output_name}.txt',index=False, sep='\t')
    print(f'Written out filtered variants to srWGS_ACAF_qc1_hwepass_vars_{output_name}.txt')
    
#Run it for each chromosome
[qc1_hwepass_vars_outputter(f'../3_output/2_VARIANT_QC/srWGS_ACAF/srWGS_ACAF_variantqc_1_{x}.bim', f'../3_output/2_VARIANT_QC/srWGS_ACAF/hwepass/{x}.txt',x) for x in [f"chr{i}" for i in range(1, 23)]]


### Double Check

In [ ]:
# #Below code cells are to double check that the function works correctly
# ancestries = ['afr', 'eur', 'eas', 'amr', 'mid', 'sas']
# #This imports in the variants which pass srWGS_variantqc1
# srwgs_variantqc1_pass_vars_chr22 = pd.read_csv('../3_output/2_VARIANT_QC/srWGS_ACAF/srWGS_ACAF_variantqc_1_chr22.bim', sep='\s+', header=None)
# print(f'Shape of reference varlist: {srwgs_variantqc1_pass_vars_chr22.shape}')
# srwgs_hwepass_vars_chr22 = [pd.read_csv(f'../3_output/2_VARIANT_QC/srWGS_ACAF/hwepass/srWGS_ACAF_hwepass_{x}_chr22.bim', sep='\s+', header=None) for x in ancestries]
# [print(x.shape) for x in srwgs_hwepass_vars_chr22]

In [ ]:
# #This contains the variants which fail the HWE filter on for each ancestry
# srwgs_hwefail_vars_chr22 = [srwgs_variantqc1_pass_vars_chr22.loc[~srwgs_variantqc1_pass_vars_chr22[1].isin(x[1])] for x in srwgs_hwepass_vars_chr22]
# [print(x.shape) for x in srwgs_hwefail_vars_chr22]

In [ ]:
# #Now find all the variants failing across all the ancestries
# srwgs_hwefail_vars_chr22_allancestries = set(srwgs_hwefail_vars_chr22[0].iloc[:, 1]).intersection(*(set(df.iloc[:, 1]) for df in srwgs_hwefail_vars_chr22[1:]))
# len(srwgs_hwefail_vars_chr22_allancestries )

### Dstat

# Upload to GCP

In [ ]:
%%capture
#Final upload of all the code and the output files from this analysis to GCP Bucket
!gsutil cp 2_VARIANT_QC.ipynb {my_bucket}/HCM_GWAS_V2/2_scripts/

In [ ]:
%%capture
!gsutil -m cp -r ../3_output/2_VARIANT_QC/array/ {my_bucket}/HCM_GWAS_V2/3_output/2_VARIANT_QC/

In [ ]:
%%capture
!gsutil -m cp -r ../3_output/2_VARIANT_QC/srWGS_ACAF/ {my_bucket}/HCM_GWAS_V2/3_output/2_VARIANT_QC/